# 3.3 Comparison of the cross-correlation (3.1) and the DLNM (3.2)

The preregistration plans the DLNM as the main analysis and the prewhitened cross-correlation (CCF) as a complementary one. PM2.5, a secondary exposure analysed exactly like H3 (PM10) in both notebooks, is compared alongside H1–H3 (as `PM25`). This notebook puts their results side by side, from the saved outputs of `03_1_cross_correlation.ipynb` and `03_2_dlnm.ipynb` (no model is refitted here), and checks first that both were run on the same panel.

**What each method measures.**

| | CCF (3.1) | DLNM (3.2) |
|---|---|---|
| Association | linear correlation of the prewhitened exposure with the prewhitened arrivals at one lag | log-RR of arrivals, non-linear in temperature (H1), linear otherwise; lag-specific and cumulative over all lags |
| Seasonality | 1 df/year trend + 3 annual harmonics (7 df/year as sensitivity) | natural spline of time, 7 df/year |
| Autocorrelation | removed by prewhitening (AR filter of the exposure applied to both series) | modelled: covariates, lag-1 residual term where needed; tests calibrated with dummy exposures |
| Influenza (H2) | weekly series, lags 0–3 weeks | daily series (weekly value on each day), lags 0–14 days |
| Scale | PM10 log, ILI log(1 + x) | PM10 linear (per 10 µg/m³), ILI log(1 + x) |
| Inference | descriptive; limits pointwise and simultaneous over lags 0 … L | confirmatory test (calibrated F-test, Holm across H1–H3) |
| Reverse direction / future exposure | negative lags (arrivals first) | future exposure term (next 1–3 days, or next week), calibrated |

The two methods therefore answer slightly different questions; agreement in direction and timing (not in the size of numbers) is what is compared. Outputs go to `outputs/03_3_comparison/`.

## Imports and configuration

In [ ]:
import json
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
CCF_DIR = ROOT / "outputs" / "03_1_cross_correlation"
DLNM_DIR = ROOT / "outputs" / "03_2_dlnm"
OUTPUTS = ROOT / "outputs" / "03_3_comparison"
OUTPUTS.mkdir(parents=True, exist_ok=True)
ALPHA = 0.05
HYPS = ("H1", "H2", "H3", "PM25")   # PM25: PM2.5, secondary exposure analysed like H3 (not in the Holm family)

COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "grey": "#8a8984", "red": "#e34948", "neutral": "#f0efec"}
plt.rcParams.update({
    "figure.figsize": (12, 3.6), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})


def fmt_p(p: float) -> str:
    return f"{p:.2g}" if p >= 1e-4 else f"{p:.1e}"

## 1. Load and check

The run stops if 3.1 and 3.2 were not run on the same panel (SHA-256 of the input panel in their run records).

In [ ]:
ri_ccf = json.loads((CCF_DIR / "run_info.json").read_text())
ri_dlnm = json.loads((DLNM_DIR / "run_info.json").read_text())
if ri_ccf["input_panel_sha256"] != ri_dlnm["input_panel_sha256"]:
    raise RuntimeError("3.1 and 3.2 were run on different panels: run both again.")
print(f"Same input panel ({ri_ccf['input_panel_sha256'][:12]}…). 3.1 run {ri_ccf['run_at']}, 3.2 run {ri_dlnm['run_at']}.")

ccf = pd.read_csv(CCF_DIR / "ccf_summary.csv", index_col="id")
ccf_values = pd.read_csv(CCF_DIR / "ccf_values.csv")
grid = pd.read_csv(CCF_DIR / "ccf_exploratory_grid.csv")
prim = pd.read_csv(DLNM_DIR / "primary_results.csv", index_col=0)
neg = pd.read_csv(DLNM_DIR / "negative_controls.csv", index_col=0)
sens = pd.read_csv(DLNM_DIR / "sensitivity.csv", index_col=0)
by_stream = pd.read_csv(DLNM_DIR / "H1_by_stream.csv", index_col=0)
lag_resp = pd.read_csv(DLNM_DIR / "lag_response.csv")

## 2. Verdict per hypothesis

- **CCF signal**: the main analysis crosses the simultaneous 95% limit at a lag ≥ 0 (exposure first), and no negative-control outcome crosses it (H2, H3).
- **DLNM signal**: calibrated, Holm-corrected p < 0.05 (the confirmatory result; for PM2.5, secondary, the calibrated p without correction).

In [ ]:
def ccf_neg_controls(h: str) -> list[str]:
    return [i for i in ccf.index if i.startswith(f"{h}_injury") and not i.endswith("no_temp")]


rows = []
for h in HYPS:
    c, d = ccf.loc[h], prim.loc[h]
    nc_cross = [i for i in ccf_neg_controls(h) if bool(ccf.loc[i, "crosses simultaneous"])]
    ccf_signal = bool(c["crosses simultaneous"]) and not nc_cross
    p_final = d["p calibrated, Holm"] if pd.notna(d["p calibrated, Holm"]) else d["p calibrated"]
    dlnm_signal = bool(p_final < ALPHA)
    if h == "H1":
        est = (f"same day: heat {d['RR_heat_lag0']:.3f} ({d['lo_heat_lag0']:.3f}–{d['hi_heat_lag0']:.3f}), "
               f"cold {d['RR_cold_lag0']:.3f} ({d['lo_cold_lag0']:.3f}–{d['hi_cold_lag0']:.3f}); cumulative vs median: heat "
               f"{d['RR_heat_vs_median']:.3f}, cold {d['RR_cold_vs_median']:.3f}")
    else:
        est = f"{d['RR']:.3f} ({d['lo']:.3f}–{d['hi']:.3f}), {d['increment']}"
    rows.append({
        "hypothesis": h,
        "CCF: peak lag (unit)": f"{int(c['peak lag'])} {c['unit']}{'s' if int(c['peak lag']) != 1 else ''}",
        "CCF: peak r (95% CI)": f"{c['peak r']:.3f} ({c['peak 95% CI'].replace(' to ', '–')})",
        "CCF: crosses simultaneous limit": bool(c["crosses simultaneous"]),
        "CCF: negative controls crossing": ", ".join(nc_cross) or "none",
        "CCF signal": ccf_signal,
        "DLNM: estimate (95% CI, calibrated)": est,
        "DLNM: calibrated p (Holm for H1–H3)": fmt_p(p_final) + ("" if pd.notna(d["p calibrated, Holm"]) else " (secondary, no Holm)"),
        "DLNM signal": dlnm_signal,
        "agree": ccf_signal == dlnm_signal,
    })
verdict = pd.DataFrame(rows).set_index("hypothesis")
verdict.to_csv(OUTPUTS / "verdict.csv")
with pd.option_context("display.max_colwidth", 120):
    display(verdict.T)

## 3. Timing: CCF by lag vs DLNM lag-specific RR

Top: CCF of the main analysis at lags ≥ 0 with its simultaneous 95% limit (orange). Bottom: DLNM lag-specific RR with √κ-widened 95% CI. Both on a scale of days (influenza CCF in weeks, plotted at 0, 7, 14, 21 days). The CCF shows *when* the prewhitened series move together; the DLNM shows the RR at each lag holding the other lags fixed. Agreement means the same sign at the same lags.

In [ ]:
fig, axes = plt.subplots(2, len(HYPS), figsize=(17, 6.5), sharex="col")
for j, h in enumerate(HYPS):
    v = ccf_values[(ccf_values["id"] == h) & (ccf_values["lag"] >= 0)]
    step = 7 if ccf.loc[h, "unit"] == "week" else 1
    ax = axes[0, j]
    ax.bar(v["lag"] * step, v["r"], width=0.7 * step if step > 1 else 0.7, color=COLORS["blue"])
    lim = v["simultaneous_limit"].iloc[0]
    ax.axhline(lim, color=COLORS["orange"], ls=":", lw=1.5); ax.axhline(-lim, color=COLORS["orange"], ls=":", lw=1.5)
    ax.axhline(0, color="black", lw=0.6)
    ax.set_title(f"{h}: CCF ({ccf.loc[h, 'exposure']} → {ccf.loc[h, 'outcome'].replace('arr_', '').replace('_mts_all', '')})", loc="left")
    ax.set_ylabel("r (prewhitened)")
    ax = axes[1, j]
    lr = lag_resp[lag_resp["hypothesis"] == h]
    for (contrast, g), col in zip(lr.groupby("contrast", sort=False), (COLORS["red"], COLORS["blue"])):
        col = COLORS["blue"] if h != "H1" else col
        ax.fill_between(g["lag_days"], g["lo"], g["hi"], color=col, alpha=0.15, lw=0)
        ax.plot(g["lag_days"], g["RR"], color=col, lw=2, marker="o" if len(g) < 6 else None, label=contrast)
    ax.axhline(1, color="black", lw=0.6)
    ax.set_xlabel("lag (days)"); ax.set_ylabel("RR (DLNM)")
    ax.set_title(f"{h}: DLNM lag-specific RR", loc="left")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUTS / "timing_ccf_vs_dlnm.png"); plt.show()

# Same-sign check at the CCF's lags (H1: heat contrast, i.e. warmer than the median)
rows = []
for h in HYPS:
    v = ccf_values[(ccf_values["id"] == h) & (ccf_values["lag"] >= 0)].set_index("lag")
    step = 7 if ccf.loc[h, "unit"] == "week" else 1
    lr = lag_resp[lag_resp["hypothesis"] == h]
    lr = (lr[lr["contrast"].str.startswith("heat")] if h == "H1" else lr).set_index("lag_days")
    for lag, r in v.iterrows():
        d = int(lag) * step
        if d in lr.index:
            rr = lr.loc[d]
            rows.append({"hypothesis": h, "lag (days)": d, "CCF r": round(r["r"], 3),
                         "CCF crosses simultaneous": abs(r["r"]) > r["simultaneous_limit"],
                         "DLNM RR (95% CI)": f"{rr['RR']:.3f} ({rr['lo']:.3f}–{rr['hi']:.3f})",
                         "DLNM CI excludes 1": (rr["lo"] > 1) or (rr["hi"] < 1),
                         "same sign": np.sign(r["r"]) == np.sign(np.log(rr["RR"]))})
timing = pd.DataFrame(rows)
timing.to_csv(OUTPUTS / "timing.csv", index=False)
# Same day vs later lags: CCF r at lag 0 and mean r over later lags; DLNM RR at lag 0 and the product of the later lag RRs
rows = []
for h in HYPS:
    v = ccf_values[(ccf_values["id"] == h) & (ccf_values["lag"] >= 0)].set_index("lag")
    lr = lag_resp[lag_resp["hypothesis"] == h]
    lr = (lr[lr["contrast"].str.startswith("heat")] if h == "H1" else lr).set_index("lag_days")
    unit = ccf.loc[h, "unit"]
    rows.append({"hypothesis": h,
                 "CCF r, lag 0": round(v.loc[0, "r"], 3),
                 "CCF lag 0 crosses simultaneous": abs(v.loc[0, "r"]) > v.loc[0, "simultaneous_limit"],
                 f"CCF mean r, later lags": round(v.loc[1:, "r"].mean(), 3),
                 "CCF later lags crossing simultaneous": ", ".join(f"{l} {unit}" for l, r in v.loc[1:].iterrows() if abs(r["r"]) > r["simultaneous_limit"]) or "none",
                 "DLNM RR, lag 0": f"{lr.loc[0, 'RR']:.3f} ({lr.loc[0, 'lo']:.3f}–{lr.loc[0, 'hi']:.3f})",
                 "DLNM RR, later lags combined": round(float(np.exp(np.log(lr.loc[1:, "RR"]).sum())), 3),
                 "DLNM RR, cumulative": round(float(np.exp(np.log(lr["RR"]).sum())), 3)})
timing_split = pd.DataFrame(rows).set_index("hypothesis")
timing_split.to_csv(OUTPUTS / "timing_split.csv")
print("Same day vs later lags (H1: warmer than the median, 99th percentile; DLNM later lags = days 1 to the maximum lag):")
display(timing_split)

## 4. Negative controls and the reverse direction

- **Outcome controls** (injuries by unit and by diagnosis, for H2, H3 and PM2.5): CCF crossing of the simultaneous limit vs the DLNM estimate (nominal p; the H3 and PM2.5 DLNM tests are calibrated, κ ≈ 1).
- **Reverse direction**: CCF negative lags (arrivals first) outside the pointwise limits vs the DLNM future-exposure term (calibrated p). Both ask whether the exposure "predicts" arrivals that came before it, which no causal effect can do.

In [ ]:
pairs = {"H2_injury_kpp": "H2 → injuries (KPP)", "H2_injury_icd": "H2 → injuries (ICD S/T)",
         "H3_injury_kpp": "H3 → injuries (KPP)", "H3_injury_icd": "H3 → injuries (ICD S/T)",
         "PM25_injury_kpp": "PM25 → injuries (KPP)", "PM25_injury_icd": "PM25 → injuries (ICD S/T)"}
rows = []
for cid, dl in pairs.items():
    c, d = ccf.loc[cid], neg.loc[dl]
    rows.append({"control": dl, "CCF peak lag, r": f"{int(c['peak lag'])} {c['unit']}, {c['peak r']:.3f}",
                 "CCF crosses simultaneous": bool(c["crosses simultaneous"]),
                 "DLNM RR (95% CI, nominal)": f"{d['RR']:.3f} ({d['lo']:.3f}–{d['hi']:.3f})", "DLNM p (nominal)": fmt_p(d["p"])})
outcome_controls = pd.DataFrame(rows).set_index("control")
rows = []
for h in HYPS:
    v = ccf_values[(ccf_values["id"] == h) & (ccf_values["lag"] < 0)]
    rows.append({"hypothesis": h, "CCF: negative lags outside pointwise": int((v["r"].abs() > v["pointwise_limit"]).sum()),
                 "CCF: of negative lags tested": len(v),
                 "CCF: negative lags crossing simultaneous": int((v["r"].abs() > v["simultaneous_limit"]).sum()),
                 "DLNM: future exposure, calibrated p": fmt_p(neg.loc[f"{h} + future exposure", "p_lead calibrated"])})
reverse = pd.DataFrame(rows).set_index("hypothesis")
outcome_controls.to_csv(OUTPUTS / "negative_controls.csv"); reverse.to_csv(OUTPUTS / "reverse_direction.csv")
display(outcome_controls); display(reverse)

## 5. Matching sensitivity analyses

Variants that exist in both notebooks. CCF: peak r at lags ≥ 0 and whether it crosses the simultaneous limit. DLNM: headline estimate and nominal p (for H1 and H2 the nominal test is liberal by κ ≈ 1.3–1.4, §6 of 3.2). For H1 the DLNM headline is the same-day RR for heat (99th percentile vs median), the counterpart of the CCF's lag-0 correlation.

In [ ]:
MATCH = [
    ("H1", "main specification", "H1", "H1 primary"),
    ("H1", "7 df/year time spline", "H1_spline7", "H1 primary"),
    ("H1", "excluding COVID period", "H1_nocovid", "H1 excluding COVID period"),
    ("H2", "main specification", "H2", "H2 primary"),
    ("H2", "7 df/year time spline", "H2_spline7", "H2 primary"),
    ("H2", "influenza index (2021–2026)", "H2_flu_index", "H2 influenza index (2021–2026)"),
    ("H2", "ILI on the raw scale", "H2_raw_scale", "H2 raw scale"),
    ("H3", "main specification (CCF log scale; DLNM linear)", "H3", "H3 primary"),
    ("H3", "PM10 log scale", "H3", "H3 log scale"),
    ("H3", "PM10 linear", "H3_raw_scale", "H3 primary"),
    ("H3", "7 df/year time spline", "H3_spline7", "H3 primary"),
    ("H3", "excluding COVID period", "H3_nocovid", "H3 excluding COVID period"),
    ("H3", "verified PM10 only", "H3_verified", "H3 verified PM10 only"),
    ("H3", "without temperature adjustment", "H3_no_temp", "H3 no temperature adjustment"),
    ("PM25", "main specification (CCF log scale; DLNM linear)", "PM25", "PM25 primary"),
    ("PM25", "PM2.5 log scale", "PM25", "PM25 log scale"),
    ("PM25", "PM2.5 linear", "PM25_raw_scale", "PM25 primary"),
    ("PM25", "7 df/year time spline", "PM25_spline7", "PM25 primary"),
    ("PM25", "excluding COVID period", "PM25_nocovid", "PM25 excluding COVID period"),
    ("PM25", "verified PM2.5 only", "PM25_verified", "PM25 verified PM2.5 only"),
    ("PM25", "without temperature adjustment", "PM25_no_temp", "PM25 no temperature adjustment"),
]
rows = []
for h, label, cid, did in MATCH:
    c, d = ccf.loc[cid], sens.loc[did]
    if h == "H1":
        dl = f"same-day heat {d['RR_heat_lag0']:.3f} ({d['lo_heat_lag0']:.3f}–{d['hi_heat_lag0']:.3f})"
        dsig = d["lo_heat_lag0"] > 1 or d["hi_heat_lag0"] < 1
    else:
        dl = f"{d['RR']:.3f} ({d['lo']:.3f}–{d['hi']:.3f})"
        dsig = d["p"] < ALPHA
    rows.append({"hypothesis": h, "variant": label, "CCF peak lag, r": f"{int(c['peak lag'])} {c['unit']}, {c['peak r']:.3f}",
                 "CCF crosses simultaneous": bool(c["crosses simultaneous"]), "DLNM estimate (nominal 95% CI)": dl,
                 "DLNM p (nominal)": fmt_p(d["p"]), "DLNM signal (nominal)": bool(dsig),
                 "agree": bool(c["crosses simultaneous"]) == bool(dsig)})
matched = pd.DataFrame(rows).set_index(["hypothesis", "variant"])
matched.to_csv(OUTPUTS / "matched_sensitivity.csv")
display(matched)
print(f"Agreement in {matched['agree'].sum()} of {len(matched)} matched variants.")

## 6. H1 by stream

CCF exploratory screen (temperature against each stream, main specification) vs the DLNM H1 model fitted to each stream (secondary, nominal CIs).

In [ ]:
streams = {"injuries (KPP)": "injury", "medical (EzB + INP)": "medical", "fast track (HP)": "fasttrack"}
rows = []
for dl, cg in streams.items():
    g = grid[(grid["exposure"] == "temp_mean") & (grid["outcome"] == cg)].iloc[0]
    d = by_stream.loc[dl]
    rows.append({"stream": dl, "CCF peak lag, r": f"{int(g['peak lag'])} day, {g['peak r']:.3f}", "CCF crosses simultaneous": bool(g["crosses"]),
                 "DLNM same-day heat": f"{d['RR_heat_lag0']:.3f} ({d['lo_heat_lag0']:.3f}–{d['hi_heat_lag0']:.3f})",
                 "DLNM same-day cold": f"{d['RR_cold_lag0']:.3f} ({d['lo_cold_lag0']:.3f}–{d['hi_cold_lag0']:.3f})",
                 "DLNM cumulative heat vs median": f"{d['RR_heat_vs_median']:.3f} ({d['lo_heat_vs_median']:.3f}–{d['hi_heat_vs_median']:.3f})",
                 "DLNM cumulative cold vs median": f"{d['RR_cold_vs_median']:.3f} ({d['lo_cold_vs_median']:.3f}–{d['hi_cold_vs_median']:.3f})"})
streams_cmp = pd.DataFrame(rows).set_index("stream")
streams_cmp.to_csv(OUTPUTS / "H1_by_stream.csv")
display(streams_cmp)

## 7. PM2.5: all arrivals and the coarse fraction

PM2.5 appears in every table above (as `PM25`). Here: PM2.5 against all arrivals (CCF screen vs DLNM, nominal), and the coarse fraction PM10 − PM2.5 (DLNM only).

In [ ]:
extra = pd.read_csv(DLNM_DIR / "pm25_extra.csv", index_col=0)
g = grid[(grid["exposure"] == "pm25") & (grid["outcome"] == "all")].iloc[0]
d = extra.loc["PM2.5 → all arrivals"]
c = extra.loc["Coarse PM (PM10 − PM2.5) → medical"]
pm25_cmp = pd.DataFrame([
    {"analysis": "PM2.5 → all arrivals", "CCF peak lag, r": f"{int(g['peak lag'])} day, {g['peak r']:.3f}", "CCF crosses simultaneous": bool(g["crosses"]),
     "DLNM RR per 10 µg/m³ (95% CI)": f"{d['RR']:.3f} ({d['lo']:.3f}–{d['hi']:.3f})", "DLNM p (nominal)": fmt_p(d["p"])},
    {"analysis": "Coarse PM (PM10 − PM2.5) → medical", "CCF peak lag, r": "not analysed", "CCF crosses simultaneous": None,
     "DLNM RR per 10 µg/m³ (95% CI)": f"{c['RR']:.3f} ({c['lo']:.3f}–{c['hi']:.3f})", "DLNM p (nominal)": fmt_p(c["p"])},
]).set_index("analysis")
pm25_cmp.to_csv(OUTPUTS / "pm25_extra.csv")
display(pm25_cmp)

## 8. Summary

Generated from the tables above, so it stays in line with the current results.

In [ ]:
lines = []
for h, r in verdict.iterrows():
    both = "both methods show an association" if r["CCF signal"] and r["DLNM signal"] else (
        "neither method shows an association" if not r["CCF signal"] and not r["DLNM signal"] else
        f"the methods disagree (CCF {'signal' if r['CCF signal'] else 'no signal'}, DLNM {'signal' if r['DLNM signal'] else 'no signal'})")
    lines.append(f"- **{h}**: {both}. CCF peak {r['CCF: peak r (95% CI)']} at lag {r['CCF: peak lag (unit)']}"
                 f"{'; negative controls crossing: ' + r['CCF: negative controls crossing'] if r['CCF: negative controls crossing'] != 'none' else ''}. "
                 f"DLNM {r['DLNM: estimate (95% CI, calibrated)']}, calibrated p = {r['DLNM: calibrated p (Holm for H1–H3)']}.")
t1 = timing_split.loc["H1"]
lines.append(f"- **Timing of H1**: both methods put the association on the same day (CCF r = {t1['CCF r, lag 0']} at lag 0; "
             f"DLNM same-day heat RR {t1['DLNM RR, lag 0']}). Over the following days the DLNM gives most of it back "
             f"(later lags combined RR {t1['DLNM RR, later lags combined']}, cumulative {t1['DLNM RR, cumulative']}), and the CCF's later lags "
             f"average r = {t1['CCF mean r, later lags']} with {t1['CCF later lags crossing simultaneous']} crossing the simultaneous limit"
             + (": a short-term shift in the day of attendance rather than additional arrivals (the cumulative RR's 95% CI, "
                f"{prim.loc['H1', 'lo_heat_vs_median']:.3f}–{prim.loc['H1', 'hi_heat_vs_median']:.3f}, includes 1)."
                if t1["DLNM RR, later lags combined"] < 1 and prim.loc["H1", "lo_heat_vs_median"] <= 1 <= prim.loc["H1", "hi_heat_vs_median"] else "."))
dis = matched[~matched["agree"]]
lines.append(f"- **Matched sensitivity analyses**: the two methods agree on signal / no signal in {int(matched['agree'].sum())} of {len(matched)}"
             + ("; they disagree for " + "; ".join(f"{h} {v} (CCF {'signal' if r['CCF crosses simultaneous'] else 'none'}, DLNM nominal p = {r['DLNM p (nominal)']})"
                                                  for (h, v), r in dis.iterrows()) if len(dis) else "") + ".")
lines.append(f"- **Reverse direction**: CCF negative lags crossing the simultaneous limit: "
             + ", ".join(f"{h} {r['CCF: negative lags crossing simultaneous']}" for h, r in reverse.iterrows())
             + "; DLNM future exposure (calibrated p): " + ", ".join(f"{h} {r['DLNM: future exposure, calibrated p']}" for h, r in reverse.iterrows()) + ".")
r = pm25_cmp.loc["PM2.5 → all arrivals"]
lines.append(f"- **PM2.5 and all arrivals**: CCF {'crosses' if r['CCF crosses simultaneous'] else 'does not cross'} ({r['CCF peak lag, r']}); "
             f"DLNM RR {r['DLNM RR per 10 µg/m³ (95% CI)']}, nominal p = {r['DLNM p (nominal)']} (driven by injuries, see the negative controls).")
from IPython.display import Markdown
display(Markdown("\n".join(lines)))
_ = (OUTPUTS / "summary.md").write_text("\n".join(lines))

## 9. Run record

In [ ]:
def git(*args) -> str:
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001
        return "unavailable"


run_info = {
    "run_at": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": bool(git("status", "--porcelain")),
    "python": platform.python_version(),
    "input_panel_sha256": ri_ccf["input_panel_sha256"],
    "input_runs": {"03_1": ri_ccf["run_at"], "03_2": ri_dlnm["run_at"]},
}
(OUTPUTS / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(f"Saved {(OUTPUTS / 'run_info.json').relative_to(ROOT)}")